# Product Performance

This notebook ranks products by (1) total recorded quantity and (2) the number of distinct invoices containing the product. It uses rows with `Quantity > 0` and `UnitPrice > 0` as a **provisional sales-like population**. This sign-based rule is not a verified business definition of a completed sale.

Descriptions are audited as labels, not used as product keys. All product metrics are grouped by `StockCode`; normalized descriptions are joined afterward so inconsistent labels cannot split or inflate product totals.

## 1. Load and preserve the source

Load the original workbook into `retail`, validate the expected columns, and keep a snapshot to check that later analysis leaves the loaded data unchanged.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "raw" / "Online Retail.xlsx"
retail = pd.read_excel(data_path, engine="openpyxl")

required_columns = {"InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"}
missing_columns = required_columns.difference(retail.columns)
assert not missing_columns, f"Required columns are missing: {sorted(missing_columns)}"
source_snapshot = retail.copy(deep=True)

print(f"Workbook: {data_path}")
print(f"Source rows: {len(retail):,}; columns: {retail.shape[1]}")
print(f"Missing StockCode: {retail['StockCode'].isna().sum():,}; missing InvoiceNo: {retail['InvoiceNo'].isna().sum():,}")

Workbook: c:\Users\pc\Desktop\omar benamara\OMAR Bureau\pricing-demand-analysis\data\raw\Online Retail.xlsx
Source rows: 541,909; columns: 8
Missing StockCode: 0; missing InvoiceNo: 0


## 2. Define the provisional population

Apply the requested positive-quantity, positive-price rule to a separate copy. The checks make population coverage explicit and catch changes in the input workbook.

In [2]:
sales_like_mask = retail["Quantity"].gt(0) & retail["UnitPrice"].gt(0)
sales_like = retail.loc[sales_like_mask].copy()

population_summary = {
    "rows": len(sales_like),
    "distinct_stock_codes": sales_like["StockCode"].nunique(),
    "distinct_invoices": sales_like["InvoiceNo"].nunique(),
    "total_quantity": sales_like["Quantity"].sum(),
    "missing_descriptions": sales_like["Description"].isna().sum(),
}
display(pd.Series(population_summary, name="value"))

assert sales_like["Quantity"].gt(0).all()
assert sales_like["UnitPrice"].gt(0).all()
assert sales_like["StockCode"].notna().all(), "Product grouping requires non-missing StockCode values."
assert sales_like["InvoiceNo"].notna().all(), "Invoice counts require non-missing InvoiceNo values."

rows                     530104
distinct_stock_codes       3922
distinct_invoices         19960
total_quantity          5588376
missing_descriptions          0
Name: value, dtype: int64

## 3. Aggregate product performance

Aggregate only by `StockCode`. Quantity is summed across qualifying rows; invoice breadth is the number of distinct qualifying `InvoiceNo` values for each stock code.

In [3]:
product_metrics = (
    sales_like.groupby("StockCode", as_index=False, dropna=False)
    .agg(
        total_quantity=("Quantity", "sum"),
        distinct_invoices=("InvoiceNo", "nunique"),
        qualifying_rows=("StockCode", "size"),
    )
)

print(f"Products in aggregated table: {len(product_metrics):,}")
print(f"Grouped quantity total: {product_metrics['total_quantity'].sum():,}")
print(f"Grouped qualifying rows: {product_metrics['qualifying_rows'].sum():,}")

Products in aggregated table: 3,922
Grouped quantity total: 5,588,376
Grouped qualifying rows: 530,104


## 4. Audit descriptions separately

For auditing only, trim whitespace, collapse repeated spaces, and case-fold descriptions. Count normalized variants per stock code and choose a display label only when its normalized form has a unique highest row count. A tie is left unlabeled as ambiguous; missing descriptions remain visible in the audit. The source descriptions are not overwritten.

In [4]:
description_rows = sales_like[["StockCode", "Description"]].copy()
description_rows["description_normalized"] = (
    description_rows["Description"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)
description_rows["description_normalized"] = description_rows["description_normalized"].replace("", pd.NA)

description_variants = (
    description_rows.dropna(subset=["description_normalized"])
    .groupby(["StockCode", "description_normalized"]).size()
    .rename("description_rows").reset_index()
)
variant_stats = description_variants.groupby("StockCode").agg(
    description_variant_count=("description_normalized", "nunique"),
    described_rows=("description_rows", "sum"),
)
missing_description_counts = sales_like.groupby("StockCode")["Description"].agg(
    description_missing_rows=lambda values: values.isna().sum()
)

if not description_variants.empty:
    top_variant_count = description_variants.groupby("StockCode")["description_rows"].transform("max")
    top_variants = description_variants.loc[description_variants["description_rows"].eq(top_variant_count)]
    top_variant_counts = top_variants.groupby("StockCode").size()
    unique_leader_codes = top_variant_counts.loc[top_variant_counts.eq(1)].index
    canonical_labels = (
        top_variants.loc[top_variants["StockCode"].isin(unique_leader_codes), ["StockCode", "description_normalized"]]
        .set_index("StockCode")["description_normalized"]
        .rename("canonical_description")
    )
else:
    canonical_labels = pd.Series(dtype="string", name="canonical_description")

description_audit = pd.DataFrame(index=pd.Index(sales_like["StockCode"].unique(), name="StockCode"))
description_audit = description_audit.join(variant_stats).join(missing_description_counts).join(canonical_labels)
description_audit["description_variant_count"] = description_audit["description_variant_count"].fillna(0).astype(int)
description_audit["described_rows"] = description_audit["described_rows"].fillna(0).astype(int)
description_audit["description_missing_rows"] = description_audit["description_missing_rows"].fillna(0).astype(int)
description_audit["description_audit_status"] = "ambiguous_top_label"
description_audit.loc[description_audit["description_variant_count"].eq(0), "description_audit_status"] = "missing_description"
description_audit.loc[description_audit["description_variant_count"].eq(1), "description_audit_status"] = "single_normalized_label"
description_audit.loc[description_audit["canonical_description"].notna() & description_audit["description_variant_count"].gt(1), "description_audit_status"] = "unique_most_frequent_label"

print("Product counts by description-audit status:")
display(description_audit["description_audit_status"].value_counts().rename_axis("status").to_frame("products"))
print(f"Qualifying rows with missing Description: {sales_like['Description'].isna().sum():,}")
print("Examples of products with multiple normalized variants or unresolved labels:")
audit_examples = description_audit.loc[
    description_audit["description_variant_count"].gt(1) | description_audit["description_audit_status"].eq("missing_description")
].head(10)
display(audit_examples.reset_index())
if not audit_examples.empty:
    display(description_variants.loc[description_variants["StockCode"].isin(audit_examples.index)].sort_values(["StockCode", "description_rows", "description_normalized"], ascending=[True, False, True]).head(30))

Product counts by description-audit status:


,products
status,
single_normalized_label,3710
unique_most_frequent_label,210
ambiguous_top_label,2


Qualifying rows with missing Description: 0
Examples of products with multiple normalized variants or unresolved labels:


,StockCode,description_variant_count,described_rows,description_missing_rows,canonical_description,description_audit_status
0,85123A,2,2265,0,white hanging heart t-light holder,unique_most_frequent_label
1,71053,2,346,0,white metal lantern,unique_most_frequent_label
2,22632,2,405,0,hand warmer red retrospot,unique_most_frequent_label
3,82486,2,616,0,wood s/3 cabinet ant white finish,unique_most_frequent_label
4,84997B,2,374,0,childrens cutlery retrospot red,unique_most_frequent_label
5,84997C,2,336,0,childrens cutlery polkadot blue,unique_most_frequent_label
6,20725,2,1595,0,lunch bag red retrospot,unique_most_frequent_label
7,21175,2,818,0,gin + tonic diet metal sign,unique_most_frequent_label
8,22778,2,148,0,glass cloche small,unique_most_frequent_label
9,22466,2,212,0,fairy tale cottage night light,unique_most_frequent_label


,StockCode,description_normalized,description_rows
101,20725,lunch bag red retrospot,1594
102,20725,lunch bag red spotty,1
342,21175,gin + tonic diet metal sign,768
343,21175,gin and tonic diet metal sign,50
1300,22466,fairy tale cottage night light,117
1301,22466,fairy tale cottage nightlight,95
1465,22632,hand warmer red retrospot,387
1464,22632,hand warmer red polka dot,18
1611,22778,glass cloche small,147
1610,22778,glass bell jar small,1


## 5. Rank products on each measure

Attach the description audit after aggregation, then show separate top-20 tables. Ties are resolved by ascending `StockCode` for reproducible display.

In [5]:
product_performance = product_metrics.merge(
    description_audit.reset_index(), on="StockCode", how="left", validate="one_to_one"
)

top_by_quantity = product_performance.sort_values(
    ["total_quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).head(20)
top_by_invoices = product_performance.sort_values(
    ["distinct_invoices", "StockCode"], ascending=[False, True], kind="mergesort"
).head(20)

result_columns = [
    "StockCode", "canonical_description", "description_audit_status",
    "total_quantity", "distinct_invoices", "qualifying_rows",
]
print("Top 20 products by total recorded quantity:")
display(top_by_quantity[result_columns].reset_index(drop=True))
print("Top 20 products by distinct qualifying invoices:")
display(top_by_invoices[result_columns].reset_index(drop=True))

Top 20 products by total recorded quantity:


,StockCode,canonical_description,description_audit_status,total_quantity,distinct_invoices,qualifying_rows
0,23843,"paper craft , little birdie",single_normalized_label,80995,1,1
1,23166,medium ceramic top storage jar,single_normalized_label,78033,247,250
2,22197,popcorn holder,unique_most_frequent_label,56921,1392,1426
3,84077,world war 2 gliders asstd designs,single_normalized_label,55047,535,537
4,85099B,jumbo bag red retrospot,single_normalized_label,48474,2089,2112
5,85123A,white hanging heart t-light holder,unique_most_frequent_label,37660,2198,2265
6,84879,assorted colour bird ornament,single_normalized_label,36461,1455,1489
7,21212,pack of 72 retrospot cake cases,single_normalized_label,36419,1320,1370
8,23084,rabbit night light,single_normalized_label,30788,994,1036
9,22492,mini paint set vintage,single_normalized_label,26633,380,380


Top 20 products by distinct qualifying invoices:


,StockCode,canonical_description,description_audit_status,total_quantity,distinct_invoices,qualifying_rows
0,85123A,white hanging heart t-light holder,unique_most_frequent_label,37660,2198,2265
1,85099B,jumbo bag red retrospot,single_normalized_label,48474,2089,2112
2,22423,regency cakestand 3 tier,single_normalized_label,13879,1988,2017
3,47566,party bunting,single_normalized_label,18295,1685,1706
4,20725,lunch bag red retrospot,unique_most_frequent_label,19553,1565,1595
5,84879,assorted colour bird ornament,single_normalized_label,36461,1455,1489
6,22197,popcorn holder,unique_most_frequent_label,56921,1392,1426
7,22720,set of 3 cake tins pantry design,single_normalized_label,7493,1385,1399
8,21212,pack of 72 retrospot cake cases,single_normalized_label,36419,1320,1370
9,22383,lunch bag suki design,single_normalized_label,12552,1284,1326


## 6. Validate aggregation and audit joins

Reconcile grouped rows and units to the filtered source population, confirm one row per product, bound invoice counts, and check that adding descriptions did not change product coverage. These checks are assertions: a mismatch should stop execution for investigation.

In [6]:
population_invoice_count = sales_like["InvoiceNo"].nunique()

assert retail.equals(source_snapshot), "The loaded source dataframe was modified during analysis."
assert product_metrics["StockCode"].is_unique, "The aggregated table has duplicate StockCode keys."
assert len(product_metrics) == sales_like["StockCode"].nunique(), "Product count does not reconcile."
assert product_metrics["total_quantity"].sum() == sales_like["Quantity"].sum(), "Grouped quantity does not reconcile."
assert product_metrics["qualifying_rows"].sum() == len(sales_like), "Grouped row counts do not reconcile."
assert product_metrics["distinct_invoices"].between(1, population_invoice_count).all(), "A product invoice count is outside valid population bounds."
assert len(product_performance) == len(product_metrics), "Description join changed product row count."
assert product_performance["StockCode"].is_unique, "Description join duplicated product keys."
assert product_performance["description_variant_count"].notna().all(), "Description audit is missing product coverage."
assert product_performance["description_missing_rows"].sum() == sales_like["Description"].isna().sum(), "Missing-description audit does not reconcile."

expected_population = {"rows": 530104, "distinct_stock_codes": 3922, "distinct_invoices": 19960, "total_quantity": 5588376}
for metric, expected_value in expected_population.items():
    actual_value = population_summary[metric]
    assert actual_value == expected_value, f"Unexpected {metric}: got {actual_value:,}, expected {expected_value:,}; inspect the source workbook and filter."

print("PASS: source frame unchanged")
print("PASS: qualifying population satisfies Quantity > 0 and UnitPrice > 0")
print("PASS: product keys, grouped rows, total quantity, and invoice bounds reconcile")
print("PASS: description audit reconciles and its join preserves product rows")
print("PASS: population matches the previously documented workbook counts")
print(f"Missing Description rows retained in population: {sales_like['Description'].isna().sum():,}")

PASS: source frame unchanged
PASS: qualifying population satisfies Quantity > 0 and UnitPrice > 0
PASS: product keys, grouped rows, total quantity, and invoice bounds reconcile
PASS: description audit reconciles and its join preserves product rows
PASS: population matches the previously documented workbook counts
Missing Description rows retained in population: 0


## 7. Main results and interpretation

The next cell summarizes the leaders for each measure directly from the ranked tables. The full top-20 tables above retain the remaining ranking detail.

In [7]:
quantity_leader = top_by_quantity.iloc[0]
invoice_leader = top_by_invoices.iloc[0]
print("Main results")
print("Highest total recorded quantity: "
      f"StockCode {quantity_leader['StockCode']} ("
      f"{quantity_leader['canonical_description'] if pd.notna(quantity_leader['canonical_description']) else 'description unresolved'}), "
      f"{quantity_leader['total_quantity']:,} units across "
      f"{quantity_leader['distinct_invoices']:,} distinct qualifying invoices.")
print("Appears in the most distinct qualifying invoices: "
      f"StockCode {invoice_leader['StockCode']} ("
      f"{invoice_leader['canonical_description'] if pd.notna(invoice_leader['canonical_description']) else 'description unresolved'}), "
      f"{invoice_leader['distinct_invoices']:,} invoices and "
      f"{invoice_leader['total_quantity']:,} recorded units.")
print("All assertions in Section 6 passed.")

Main results
Highest total recorded quantity: StockCode 23843 (paper craft , little birdie), 80,995 units across 1 distinct qualifying invoices.
Appears in the most distinct qualifying invoices: StockCode 85123A (white hanging heart t-light holder), 2,198 invoices and 37,660 recorded units.
All assertions in Section 6 passed.


### Limitations

- Total quantity is recorded positive-line volume under a provisional sign filter. It is not net demand, unmet demand, or proof of completed sales; returns are excluded, and stock availability is not observed. Pack sizes and bulk purchases can also make unit totals hard to compare.
- Distinct invoice count measures how broadly a product appears across qualifying invoices, not how many distinct customers bought it or how many independent demand events occurred. Invoice splitting and basket composition can affect the count.
- The workbook covers a finite historical window. These rankings describe only this observed period and population; they do not establish future demand or product performance outside it.
- Description normalization is a lightweight label audit. Case and whitespace differences are collapsed, unresolved top-count ties are left ambiguous, and normalized labels do not validate that a stock code represents a single stable product.

## StockCode 23843: broad rows and invoice

In [8]:
target_code = "23843"
target_mask = retail["StockCode"].astype("string").str.strip().eq(target_code)
target_rows = retail.loc[target_mask]
print(f"All broad retail rows for StockCode {target_code}: {len(target_rows)}")
display(target_rows)

target_sales = sales_like.loc[sales_like["StockCode"].astype("string").str.strip().eq(target_code)]
assert not target_sales.empty, f"No qualifying row found for StockCode {target_code}."
original_line = target_sales.sort_values(["Quantity", "InvoiceNo"], ascending=[False, True], kind="mergesort").iloc[0]
original_invoice = original_line["InvoiceNo"]
invoice_rows = retail.loc[retail["InvoiceNo"].eq(original_invoice)].copy()
invoice_total = (invoice_rows["Quantity"] * invoice_rows["UnitPrice"]).sum()
print(f"Selected qualifying line: InvoiceNo={original_invoice}; Quantity={original_line['Quantity']}; UnitPrice={original_line['UnitPrice']}")
print(f"Full invoice: {len(invoice_rows)} rows; total Quantity={invoice_rows['Quantity'].sum()}; total Quantity*UnitPrice={invoice_total}")
display(invoice_rows.assign(LineAmount=invoice_rows["Quantity"] * invoice_rows["UnitPrice"]))

All broad retail rows for StockCode 23843: 2


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


Selected qualifying line: InvoiceNo=581483; Quantity=80995; UnitPrice=2.08
Full invoice: 1 rows; total Quantity=80995; total Quantity*UnitPrice=168469.6


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineAmount
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.6


## StockCode 23843: negative-quantity search

In [9]:
negative_rows = retail.loc[target_mask & retail["Quantity"].lt(0)].copy()
print(f"Negative-quantity rows for StockCode {target_code}: {len(negative_rows)}")
display(negative_rows)
print(f"Any Quantity == -80995: {negative_rows['Quantity'].eq(-80995).any()}")

large_negative_rows = negative_rows.loc[negative_rows["Quantity"].abs().ge(original_line["Quantity"] * 0.5)].copy()
large_negative_rows["CustomerID_matches_original"] = large_negative_rows["CustomerID"].eq(original_line["CustomerID"])
large_negative_rows["date_after_original"] = large_negative_rows["InvoiceDate"].gt(original_line["InvoiceDate"])
print(f"Rows with abs(Quantity) >= 50% of original ({original_line['Quantity']}): {len(large_negative_rows)}")
display(large_negative_rows)

Negative-quantity rows for StockCode 23843: 1


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


Any Quantity == -80995: True
Rows with abs(Quantity) >= 50% of original (80995): 1


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,CustomerID_matches_original,date_after_original
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,True,True


## Quantity context for StockCodes 23843 and 23166

In [10]:
print("10 largest Quantity lines in sales_like:")
display(sales_like.nlargest(10, "Quantity"))
less_percentile = sales_like["Quantity"].lt(80995).mean() * 100
at_or_below_percentile = sales_like["Quantity"].le(80995).mean() * 100
print(f"80995 empirical percentile in sales_like Quantity: {less_percentile:.6f}% strictly below; {at_or_below_percentile:.6f}% at or below.")

code_23166_mask = sales_like["StockCode"].astype("string").str.strip().eq("23166")
sales_23166 = sales_like.loc[code_23166_mask]
assert not sales_23166.empty, "No qualifying rows found for StockCode 23166."
print("5 largest qualifying lines for StockCode 23166:")
display(sales_23166.nlargest(5, "Quantity"))
quantity_23166_total = sales_23166["Quantity"].sum()
quantity_23166_largest = sales_23166["Quantity"].max()
print(f"StockCode 23166 total qualifying Quantity={quantity_23166_total}; largest line={quantity_23166_largest}; largest-line share={quantity_23166_largest / quantity_23166_total:.6%}")

10 largest Quantity lines in sales_like:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,2011-10-27 12:26:00,0.21,12901.0,United Kingdom
206121,554868,22197,SMALL POPCORN HOLDER,4300,2011-05-27 10:52:00,0.72,13135.0,United Kingdom
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,2011-02-22 10:43:00,0.82,18087.0,United Kingdom
270885,560599,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,2011-07-19 17:04:00,0.06,14609.0,United Kingdom
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749.0,United Kingdom
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749.0,United Kingdom
433788,573995,16014,SMALL CHINESE STYLE SCISSOR,3000,2011-11-02 11:24:00,0.32,16308.0,United Kingdom
4945,536830,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,2880,2010-12-02 17:38:00,0.18,16754.0,United Kingdom


80995 empirical percentile in sales_like Quantity: 99.999811% strictly below; 100.000000% at or below.


5 largest qualifying lines for StockCode 23166:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
285367,561901,23166,MEDIUM CERAMIC TOP STORAGE JAR,288,2011-07-31 15:42:00,1.25,14156.0,EIRE
194462,553607,23166,MEDIUM CERAMIC TOP STORAGE JAR,240,2011-05-18 10:47:00,1.04,16684.0,United Kingdom
276513,561051,23166,MEDIUM CERAMIC TOP STORAGE JAR,144,2011-07-24 13:11:00,1.04,16684.0,United Kingdom
186770,552882,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,2011-05-12 10:10:00,1.04,14646.0,Netherlands


StockCode 23166 total qualifying Quantity=78033; largest line=74215; largest-line share=95.107198%


## Top-20 ranking sensitivity

In [11]:
original_totals = sales_like.groupby("StockCode")["Quantity"].sum()
ranking_a = product_metrics[["StockCode", "total_quantity"]].sort_values(
    ["total_quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)

largest_line_indexes = sales_like.groupby("StockCode")["Quantity"].idxmax()
without_largest_totals = (
    sales_like.drop(index=largest_line_indexes).groupby("StockCode")["Quantity"].sum()
    .reindex(original_totals.index, fill_value=0)
)
ranking_b = without_largest_totals.rename("quantity_excluding_one_largest_line").reset_index()
ranking_b = ranking_b.sort_values(
    ["quantity_excluding_one_largest_line", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)

unmatched_net_totals = retail.groupby("StockCode", dropna=False)["Quantity"].sum()
ranking_c = unmatched_net_totals.rename("unmatched_net_quantity").reset_index()
ranking_c = ranking_c.sort_values(
    ["unmatched_net_quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)

rankings = {
    "(a) Original ranking": (ranking_a, "total_quantity"),
    "(b) Excluding each product's single largest line": (ranking_b, "quantity_excluding_one_largest_line"),
    "(c) Unmatched net (signed Quantity over all retail rows)": (ranking_c, "unmatched_net_quantity"),
}
baseline_top20 = set(ranking_a.head(20)["StockCode"])

def rank_for_stock_code(frame, stock_code):
    matches = frame["StockCode"].astype("string").str.strip().eq(stock_code)
    return int(frame.index[matches][0] + 1) if matches.any() else None

for label, (ranking, quantity_column) in rankings.items():
    print(label)
    display(ranking.head(20))
    current_top20 = set(ranking.head(20)["StockCode"])
    print(f"Rank 23843: {rank_for_stock_code(ranking, '23843')}; rank 23166: {rank_for_stock_code(ranking, '23166')}")
    print(f"Original top-20 products remaining: {len(baseline_top20.intersection(current_top20))} of 20")

(a) Original ranking


,StockCode,total_quantity
0,23843,80995
1,23166,78033
2,22197,56921
3,84077,55047
4,85099B,48474
5,85123A,37660
6,84879,36461
7,21212,36419
8,23084,30788
9,22492,26633


Rank 23843: 1; rank 23166: 2
Original top-20 products remaining: 20 of 20
(b) Excluding each product's single largest line


,StockCode,quantity_excluding_one_largest_line
0,22197,52621
1,84077,50247
2,85099B,47274
3,85123A,35730
4,21212,34979
5,84879,33581
6,23084,28388
7,22492,25239
8,22616,24839
9,22178,23344


Rank 23843: 3798; rank 23166: 318
Original top-20 products remaining: 18 of 20
(c) Unmatched net (signed Quantity over all retail rows)


,StockCode,unmatched_net_quantity
0,22197,56450
1,84077,53847
2,85099B,47363
3,85123A,38830
4,84879,36221
5,21212,36039
6,23084,30646
7,22492,26437
8,22616,26315
9,21977,24753


Rank 23843: 3776; rank 23166: 377
Original top-20 products remaining: 18 of 20


## Validation checks

In [12]:
validation_results = []

def report_assertion(label, condition):
    try:
        assert condition, label
    except AssertionError as error:
        print(f"FAIL: {error}")
        validation_results.append(False)
    else:
        print(f"PASS: {label}")
        validation_results.append(True)

raw_23843_mask = (
    retail["StockCode"].astype("string").str.strip().eq("23843")
    & retail["Quantity"].gt(0)
    & retail["UnitPrice"].gt(0)
)
raw_23843_total = retail.loc[raw_23843_mask, "Quantity"].sum()
metric_23843_rows = product_metrics.loc[
    product_metrics["StockCode"].astype("string").str.strip().eq("23843"), "total_quantity"
]
report_assertion(
    "23843 qualifying total recomputed from raw retail rows equals product_metrics",
    len(metric_23843_rows) == 1 and raw_23843_total == metric_23843_rows.iloc[0],
)

top_a_for_check = ranking_a.head(20)[["StockCode", "total_quantity"]].reset_index(drop=True)
top_by_quantity_for_check = top_by_quantity[["StockCode", "total_quantity"]].reset_index(drop=True)
report_assertion("(a) original top-20 ranking equals top_by_quantity", top_a_for_check.equals(top_by_quantity_for_check))
report_assertion("retail equals source_snapshot", retail.equals(source_snapshot))
assert all(validation_results), "One or more validation checks failed."

PASS: 23843 qualifying total recomputed from raw retail rows equals product_metrics
PASS: (a) original top-20 ranking equals top_by_quantity


PASS: retail equals source_snapshot


## Observations

- Broad `retail` contains two rows for StockCode 23843: `+80995` on invoice `581483` at 09:15 and `-80995` on invoice `C581484` at 09:27 on 2011-12-09. Both rows have CustomerID 16446. The qualifying invoice has 1 row, total Quantity 80995, and total `Quantity*UnitPrice` 168469.60.
- StockCode 23843's qualifying Quantity is the largest `sales_like` line. For StockCode 23166, its largest qualifying line is 74215 of 78033 total (95.1072%).
- StockCode 23843 and 23166 rank 1 and 2 in the original ranking; after excluding one largest line they rank 3798 and 318; in the **unmatched net** all-retail ranking they rank 3776 and 377. Eighteen original top-20 products remain in each sensitivity ranking.
- These checks describe recorded rows and ranking changes; they do not establish why either quantity was recorded.

## Product quantity and return-rate rankings

Using only rows with `UnitPrice > 0`, calculate gross positive quantity, returned quantity as a positive amount, net quantity, and return rate by `StockCode`. This section reads from `retail` and does not modify it.

In [11]:
unit_price_rows = retail.loc[retail["UnitPrice"].gt(0), ["StockCode", "Quantity"]].copy()

product_returns = (
    unit_price_rows.assign(
        gross_qty=unit_price_rows["Quantity"].clip(lower=0),
        returned_qty=-unit_price_rows["Quantity"].clip(upper=0),
    )
    .groupby("StockCode", as_index=False, dropna=False)
    .agg(
        gross_qty=("gross_qty", "sum"),
        returned_qty=("returned_qty", "sum"),
        signed_qty=("Quantity", "sum"),
    )
)
product_returns["net_qty"] = product_returns["gross_qty"] - product_returns["returned_qty"]
product_returns["return_rate"] = (
    product_returns["returned_qty"]
    .div(product_returns["gross_qty"])
    .where(product_returns["gross_qty"].gt(0))
)

assert product_returns["net_qty"].eq(product_returns["signed_qty"]).all(), (
    "Per-StockCode gross quantity minus returned quantity must equal signed Quantity "
    "for rows with UnitPrice > 0."
 )
assert product_returns["net_qty"].sum() == unit_price_rows["Quantity"].sum(), (
    "Aggregated net quantity must equal the signed Quantity sum for rows with UnitPrice > 0."
 )
assert retail.equals(source_snapshot), "The loaded retail frame was modified during this analysis."

ranking_columns = ["StockCode", "gross_qty", "returned_qty", "net_qty", "return_rate"]
top20_by_net = product_returns.sort_values(
    ["net_qty", "StockCode"], ascending=[False, True], kind="mergesort"
 ).head(20)
top20_by_gross = product_returns.sort_values(
    ["gross_qty", "StockCode"], ascending=[False, True], kind="mergesort"
 ).head(20)
eligible_for_return_rate = product_returns.loc[product_returns["gross_qty"].ge(500)]
top20_by_return_rate = eligible_for_return_rate.sort_values(
    ["return_rate", "StockCode"], ascending=[False, True], kind="mergesort"
 ).head(20)

net_gross_top20_overlap = len(
    set(top20_by_net["StockCode"]).intersection(top20_by_gross["StockCode"])
 )
print("Top 20 products by net quantity:")
display(top20_by_net[ranking_columns].reset_index(drop=True))
print("Top 20 return rates among products with gross_qty >= 500:")
display(top20_by_return_rate[ranking_columns].reset_index(drop=True))
print(
    "Products shared by the top 20 net-quantity and top 20 gross-quantity rankings: "
    f"{net_gross_top20_overlap} of 20"
 )
print("PASS: per-StockCode net quantity reconciles to signed UnitPrice > 0 quantity")
print("PASS: retail remains equal to source_snapshot")

Top 20 products by net quantity:


,StockCode,gross_qty,returned_qty,net_qty,return_rate
0,22197,56921,471,56450,0.008275
1,84077,55047,1200,53847,0.021800
2,85099B,48474,1115,47359,0.023002
3,84879,36461,80,36381,0.002194
4,21212,36419,380,36039,0.010434
5,85123A,37660,2578,35082,0.068455
6,23084,30788,108,30680,0.003508
7,22492,26633,196,26437,0.007359
8,22616,26135,24,26111,0.000918
9,21977,24854,101,24753,0.004064


Top 20 return rates among products with gross_qty >= 500:


,StockCode,gross_qty,returned_qty,net_qty,return_rate
0,23843,80995,80995,0,1.000000
1,84347,9476,9376,100,0.989447
2,23166,78033,74494,3539,0.954647
3,23113,1154,946,208,0.819757
4,23055,658,432,226,0.656535
5,23056,966,579,387,0.599379
6,47566B,2382,1424,958,0.597817
7,M,7224,4066,3158,0.562846
8,23057,776,433,343,0.557990
9,22920,3427,1527,1900,0.445579


Products shared by the top 20 net-quantity and top 20 gross-quantity rankings: 18 of 20
PASS: per-StockCode net quantity reconciles to signed UnitPrice > 0 quantity
PASS: retail remains equal to source_snapshot


## StockCode 84347: large quantity rows

Show rows with `abs(Quantity) >= 1000`, then summarize the positive and negative row counts and check whether the largest quantities in each direction cancel.

In [12]:
target_stock_code = "84347"
large_quantity_mask = (
    retail["StockCode"].astype("string").str.strip().eq(target_stock_code)
    & retail["Quantity"].abs().ge(1000)
 )
large_quantity_rows = retail.loc[
    large_quantity_mask,
    ["InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice", "CustomerID"],
].sort_values("InvoiceDate", kind="mergesort")

display(large_quantity_rows.reset_index(drop=True))
positive_rows = large_quantity_rows.loc[large_quantity_rows["Quantity"].gt(0)]
negative_rows = large_quantity_rows.loc[large_quantity_rows["Quantity"].lt(0)]
largest_positive_quantity = positive_rows["Quantity"].max() if not positive_rows.empty else None
largest_negative_quantity = negative_rows["Quantity"].min() if not negative_rows.empty else None
largest_quantities_cancel = (
    largest_positive_quantity is not None
    and largest_negative_quantity is not None
    and largest_positive_quantity == abs(largest_negative_quantity)
 )
print(f"Positive rows: {len(positive_rows)}")
print(f"Negative rows: {len(negative_rows)}")
print(f"Largest positive Quantity: {largest_positive_quantity}")
print(f"Largest negative Quantity: {largest_negative_quantity}")
print(f"Largest positive and negative quantities cancel: {largest_quantities_cancel}")

,InvoiceNo,InvoiceDate,Quantity,UnitPrice,CustomerID
0,C536757,2010-12-02 14:23:00,-9360,0.03,15838.0


Positive rows: 0
Negative rows: 1
Largest positive Quantity: None
Largest negative Quantity: -9360
Largest positive and negative quantities cancel: False


## StockCode 84347: customer, chronology, and prices

Inspect all rows for CustomerID 15838, the product's 10 earliest rows, and its UnitPrice distribution. Then count products with a negative-quantity row dated before their first positive-quantity row.

In [13]:
product_code_mask = retail["StockCode"].astype("string").str.strip().eq("84347")
product_84347_rows = retail.loc[product_code_mask].sort_values(
    "InvoiceDate", kind="mergesort"
 )
customer_15838_rows = product_84347_rows.loc[product_84347_rows["CustomerID"].eq(15838)]

print("All rows for StockCode 84347 and CustomerID 15838:")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(customer_15838_rows)

print("10 earliest rows for StockCode 84347:")
display(product_84347_rows.head(10))

product_price_summary = product_84347_rows["UnitPrice"].agg(["median", "min", "max"])
print(
    "StockCode 84347 UnitPrice: "
    f"median={product_price_summary['median']}, "
    f"min={product_price_summary['min']}, "
    f"max={product_price_summary['max']}"
 )

dated_rows = retail.dropna(subset=["StockCode", "InvoiceDate"])
first_positive_dates = (
    dated_rows.loc[dated_rows["Quantity"].gt(0)]
    .groupby("StockCode")["InvoiceDate"]
    .min()
 )
first_negative_dates = (
    dated_rows.loc[dated_rows["Quantity"].lt(0)]
    .groupby("StockCode")["InvoiceDate"]
    .min()
 )
negative_before_first_positive = first_negative_dates.reindex(first_positive_dates.index).lt(
    first_positive_dates
 ).fillna(False)
products_with_negative_before_first_positive = int(negative_before_first_positive.sum())
print(
    "Products with a negative-quantity row before their first positive-quantity row: "
    f"{products_with_negative_before_first_positive}"
 )
assert retail.equals(source_snapshot), "The loaded retail frame was modified during this analysis."
print("PASS: retail remains equal to source_snapshot")

All rows for StockCode 84347 and CustomerID 15838:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
4287,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom


10 earliest rows for StockCode 84347:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
476,536408,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,6,2010-12-01 11:41:00,2.55,14307.0,United Kingdom
754,536446,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,12,2010-12-01 12:15:00,2.55,15983.0,United Kingdom
890,536488,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,1,2010-12-01 12:31:00,2.55,17897.0,United Kingdom
891,536488,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,1,2010-12-01 12:31:00,2.55,17897.0,United Kingdom
913,536488,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,3,2010-12-01 12:31:00,2.55,17897.0,United Kingdom
1110,536527,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,6,2010-12-01 13:04:00,2.55,12662.0,Germany
1762,536544,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,4,2010-12-01 14:32:00,5.06,NaN,United Kingdom
2943,536592,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,1,2010-12-01 17:06:00,5.06,NaN,United Kingdom
3738,536667,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,6,2010-12-02 12:09:00,2.55,15260.0,United Kingdom
4141,536749,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,1,2010-12-02 13:49:00,2.55,17976.0,United Kingdom


StockCode 84347 UnitPrice: median=2.55, min=0.03, max=5.95
Products with a negative-quantity row before their first positive-quantity row: 75
PASS: retail remains equal to source_snapshot
